# Session 5 — Correlation and Covariance

**Goal:** map how the inputs relate to the outcome and to each other — and learn the
four ways a correlation coefficient lies: by measuring the wrong kind of relationship,
by being dominated by outliers, by hiding redundancy between inputs, and by reversing
inside subgroups.

## What this stage does for the system

Session 4 validated that the registry represents its population. This session asks the
first modelling question: **which of the thirteen measurements carry information about
disease, and which are duplicates of each other?**

Both halves matter, and for different downstream reasons. Input-to-target correlations
give Session 9 its candidate predictors and Sessions 6-8 their hypotheses to test.
Input-to-input correlations are the quieter problem: two inputs measuring the same
underlying thing do not break a model's *predictions*, but they make its *coefficients*
unstable and uninterpretable, which is fatal for a clinical tool whose whole selling
point is being explainable. Session 9 hits that directly.

The discipline this session installs: **a correlation coefficient is a summary of a
scatter plot, and you do not get to skip looking at the scatter plot.**

## The dataset

Every session in this module works on one registry: the UCI **Heart Disease**
dataset (Cleveland), fetched live from the UCI ML Repository with `ucimlrepo` so the
notebooks are runnable by anyone without a CSV sitting on their machine. It holds 303
patients with clinical measurements (`age`, `trestbps` resting blood pressure, `chol`
serum cholesterol, `thalach` max heart rate achieved, `oldpeak` ST depression),
categorical findings (`sex`, `cp` chest-pain type, `fbs` fasting blood sugar > 120,
`restecg`, `exang` exercise-induced angina, `slope`, `ca`, `thal`), and the outcome
`num` — angiographic disease severity 0-4, which this module binarises into
`target` (0 = no disease, 1 = disease present).

Deliberately one dataset throughout: switching datasets between topics would mean
re-learning the data every session instead of building cumulative familiarity with
one problem, the way a real analyst does.

## How to read this notebook

Every code cell is followed by a short **Observe / Infer** note: *Observe* points at
exactly what to look at in that cell's output, and *Infer* explains what conclusion to
draw from it — and what a different result would imply. Read them before running the
next cell; several of them flag things worth double-checking before you move on.

## Prerequisites

This session runs entirely locally — no account or credentials needed.

```bash
pip install ucimlrepo pandas numpy scipy scikit-learn statsmodels matplotlib seaborn
```

## Step 1 — Load the registry from the UCI repository

Fetching directly from the UCI ML Repository keeps this notebook runnable by anyone,
instead of depending on a CSV already sitting on your machine. The same nine lines
open every session in this module, so the 297 patients below are the identical 297
patients every other notebook analyses.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

heart_disease = fetch_ucirepo(id=45)
df = pd.concat([heart_disease.data.features, heart_disease.data.targets], axis=1)

# `num` is severity 0-4; this module screens for disease presence, so binarise it.
df = df.dropna().reset_index(drop=True)
df["target"] = (df["num"] > 0).astype(int)
df = df.drop(columns="num")

print(f"{len(df)} patients, {len(df.columns)} columns")
print(f"disease prevalence: {df['target'].mean():.3f}")
df.head()

**Observe:** `297 patients, 14 columns` and `disease prevalence: 0.461`. The preview
shows `age`, `sex`, `cp`, `trestbps`, `chol`, `fbs`, `restecg`, `thalach`, `exang`,
`oldpeak`, `slope`, `ca`, `thal`, and the `target` column just derived.
**Infer:** 303 rows are fetched and 297 survive `dropna()` — six patients are missing
`ca` (number of major vessels seen on fluoroscopy) or `thal`. Dropping six rows out of
303 is defensible here and keeps every notebook in this module working on the identical
297 patients; on a larger fraction of missing values you would have to impute instead,
and *that* choice would itself need the distribution work of Session 3. If your row
count is not 297, you are on a different subset than every number quoted below.

## Step 2 — Rank every input by its correlation with the outcome

Pearson's $r$ measures *linear* association on a −1 to +1 scale. Computing it against
`target` for all thirteen inputs at once is the crudest useful feature screen, and its
crudeness is instructive: `target` is binary and several inputs are categorical codes,
so some of these numbers are less meaningful than they look.

In [ ]:
correlations = df.corr()["target"].drop("target").sort_values(key=abs, ascending=False)
print(correlations.round(3).to_string())

**Observe:** `thal` `+0.527`, `ca` `+0.463`, `oldpeak` `+0.424`, `thalach` `−0.424`,
`exang` `+0.421`, `cp` `+0.409` at the top; `trestbps` `+0.153`, `chol` `+0.080`, and
`fbs` `+0.003` at the bottom.
**Infer:** the ranking confirms Session 2's group-gap ordering (`thalach` and `oldpeak`
strong, `chol` weak) and Session 1's independence finding (`fbs` at three-thousandths
is as close to no relationship as this dataset produces). But two entries at the top
are not what they appear. `thal` is coded 3/6/7 and `cp` 1/4 — those are *category
labels*, and Pearson's r on them measures whether the arbitrary numeric ordering
happens to line up with disease, not whether the variable is informative. Reordering
the codes would change `thal`'s correlation while changing nothing about the data. For
those columns the correlation is a placeholder, and Session 8's chi-square test is the
right instrument. `ca` (a genuine count) and the continuous columns are fine to read
literally.

## Step 3 — Covariance vs. correlation: the same idea, different units

Covariance is the raw quantity: the average product of two variables' deviations from
their means. Correlation is covariance divided by both standard deviations, which
strips the units off and bounds the result to [−1, +1].

In [ ]:
pairs = [("age", "trestbps"), ("age", "chol"), ("age", "thalach")]

print(f"{'pair':22} {'covariance':>14} {'correlation':>13}")
for a, b in pairs:
    cov = df[[a, b]].cov().iloc[0, 1]
    corr = df[a].corr(df[b])
    print(f"{a + ' ~ ' + b:22} {cov:14.2f} {corr:13.3f}")

# Covariance is not comparable across pairs: rescale one variable and watch it move.
df_scaled = df.assign(chol_mmol=df["chol"] / 38.67)   # mg/dL -> mmol/L
print()
print(f"cov(age, chol in mg/dL)  = {df[['age', 'chol']].cov().iloc[0, 1]:8.2f}")
print(f"cov(age, chol in mmol/L) = {df_scaled[['age', 'chol_mmol']].cov().iloc[0, 1]:8.2f}")
print(f"corr, either way         = {df['age'].corr(df['chol']):8.3f} / {df_scaled['age'].corr(df_scaled['chol_mmol']):.3f}")

**Observe:** covariances of wildly different magnitudes (`46.69` for age~trestbps,
`95.36` for age~chol, `−81.92` for age~thalach) whose correlations are all in the same
narrow band; and the age~chol covariance shrinking from `95.36` to `2.47` when
cholesterol is re-expressed in mmol/L while the correlation does not move at all.
**Infer:** covariance tells you the *sign* of a relationship reliably and its
*strength* not at all, because its magnitude depends on the units of both variables.
That makes it useless for ranking pairs — which is the entire job here — and is why
every table below reports correlation. Covariance is not obsolete though: it is the
quantity that actually appears in the mathematics (Session 2's variance-of-a-sum, the
covariance matrix behind PCA), whereas correlation is the version built for human
comparison.

## Step 4 — Pearson vs. Spearman: which kind of relationship are you measuring?

Pearson measures *linear* association and is computed on the raw values, so a few
extreme points can dominate it. Spearman replaces each value by its rank, then runs
Pearson on the ranks — so it measures *monotone* association and is insensitive to
outliers and to skew. Session 3 flagged `chol` and `oldpeak` as skewed with extreme
tails; those are precisely where the two coefficients should disagree.

In [ ]:
import numpy as np
from scipy import stats

continuous = ["age", "trestbps", "chol", "thalach", "oldpeak"]
rows = []
for col in continuous:
    if col == "age":
        continue
    rows.append({
        "pair": f"age ~ {col}",
        "pearson": stats.pearsonr(df["age"], df[col]).statistic,
        "spearman": stats.spearmanr(df["age"], df[col]).statistic,
    })
comparison = pd.DataFrame(rows)
comparison["gap"] = comparison["pearson"] - comparison["spearman"]
print(comparison.round(3).to_string(index=False))

**Observe:** `age ~ thalach` agrees almost exactly between the two methods
(`−0.395` vs `−0.393`) and `age ~ trestbps` nearly as closely; the largest gap is
`age ~ oldpeak`, where Spearman's `0.252` exceeds Pearson's `0.197`, and `age ~ chol`
tilts the other way (`0.203` vs `0.183`).
**Infer:** none of these gaps is large enough to change a decision — this registry has
no pair where the two methods tell different stories — but their directions are
readable and match Session 3's shape findings. Spearman above Pearson on `oldpeak`
says the relationship with age is monotone but not straight-line, which is what a
zero-inflated column produces; Pearson above Spearman on `chol` says the linear fit is
being helped along by the extreme values Session 2 flagged. Close agreement is evidence
that a relationship is genuinely linear and not
being driven by a handful of extreme points — which is why the `age ~ thalach` pair is
safe to hand straight to Session 9's linear regression. A large gap is a flag, and its
*direction* says which way: Spearman noticeably above Pearson means a real monotone
relationship that is not straight-line, so a transform (Session 3's `log`) or a
non-linear model is needed; Pearson above Spearman means outliers are inflating the
linear fit. The practical default when the two disagree is Spearman, since it is
answering the more robust question — and note Spearman needs no distributional
assumption at all, which is the same logic behind Session 7's Mann-Whitney U.

## Step 5 — Plot before you trust the number

Anscombe's quartet is the classic demonstration that wildly different scatter plots
can share one correlation coefficient. Here, the same lesson on real columns.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (a, b) in zip(axes, [("age", "thalach"), ("age", "chol"), ("chol", "thalach")]):
    ax.scatter(df[a], df[b], alpha=0.45, s=22, color="steelblue", edgecolor="none")
    r = df[a].corr(df[b])
    m, c = np.polyfit(df[a], df[b], 1)
    xs = np.linspace(df[a].min(), df[a].max(), 50)
    ax.plot(xs, m * xs + c, "r-", lw=2)
    ax.set_xlabel(a); ax.set_ylabel(b)
    ax.set_title(f"{a} vs {b}   r = {r:+.3f}")
plt.tight_layout()
plt.show()

**Observe:** `age vs thalach` is a genuine downward cloud around its fitted line;
`age vs chol` is a near-shapeless blob with a faint upward tilt and a handful of points
stranded far above everything else; `chol vs thalach` has an r indistinguishable from
zero and a flat line through a formless scatter.
**Infer:** the middle panel is the cautionary one. Its `r = 0.203` is not zero and
would survive a naive "keep anything above 0.2" screening rule, but the plot shows a
relationship carried substantially by a few high-cholesterol patients rather than by a
consistent trend — and Session 2 identified exactly four patients above |z| = 3 on that
column. The right-hand panel shows the opposite failure mode is possible too: an r near
zero rules out a *linear* relationship and nothing else; a U-shape would produce the
same coefficient. Neither conclusion is available from the number alone, which is the
point of the step.

## Step 6 — The correlation matrix and multicollinearity

Every pairwise correlation among the inputs at once. Then the check that pairwise
correlations cannot do: the **variance inflation factor**, which detects an input
predictable from a *combination* of the others even when no single pair looks alarming.
VIF above 5 is a warning, above 10 a serious problem.

In [ ]:
import seaborn as sns

corr_matrix = df[continuous].corr()
fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(corr_matrix, annot=True, fmt="+.2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("Correlations among the continuous inputs")
plt.tight_layout()
plt.show()

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

def vif_table(frame):
    X = sm.add_constant(frame)
    return pd.DataFrame({
        "input": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    }).query("input != 'const'").round(2)

print("VIF, continuous inputs as they stand:")
print(vif_table(df[continuous]).to_string(index=False))

**Observe:** no off-diagonal correlation exceeds `|0.40|`, and every VIF sits between
`1.06` and `1.35` — comfortably below the warning threshold.
**Infer:** these five inputs are close to non-redundant, so Session 9 can fit a
multivariate regression on them and read its coefficients without the instability
multicollinearity causes. That is a real (if unexciting) finding, and worth stating
explicitly rather than skipping past — a clean diagnostic is only informative if you
also know what a dirty one looks like, which is the next cell. Note that the VIF
constant term is excluded: including it produces a huge, meaningless value, because a
column of ones is trivially "predictable" from the intercept.

In [ ]:
# What redundancy looks like: a second lab reporting the same cholesterol with noise.
rng = np.random.default_rng(0)
redundant = df[continuous].assign(
    chol_lab2=df["chol"] + rng.normal(0, 5, len(df))
)

print(f"corr(chol, chol_lab2) = {redundant['chol'].corr(redundant['chol_lab2']):.4f}")
print()
print(vif_table(redundant).to_string(index=False))

**Observe:** the two cholesterol columns correlate at about `0.995`, and their VIFs
explode into the hundreds while every other input's VIF is essentially unchanged.
**Infer:** the localisation is the useful part — multicollinearity does not degrade a
model globally, it destroys the interpretability of *the involved coefficients
specifically*. With two near-identical inputs, the fit can put any weights on them that
sum correctly, including large ones of opposite sign, so the coefficients become
unstable across resamples while predictions stay fine. For a screening tool meant to
explain itself, that is the failure that matters. The fixes, in order of preference:
drop one, average them into a single input, or use a penalised regression (ridge) that
splits the weight between them by construction. Real registries acquire this problem
constantly — the same measurement in two units, a total and its components, a score and
its inputs.

## Step 7 — Confounding: a pooled correlation that misstates every subgroup

Simpson's paradox in its practical form. A relationship measured on pooled data can be
stronger, weaker, or reversed relative to the same relationship measured inside each
subgroup, whenever a third variable drives both.

In [ ]:
pooled = df["thalach"].corr(df["oldpeak"])
within = df.groupby("target").apply(
    lambda g: g["thalach"].corr(g["oldpeak"]), include_groups=False
)

print(f"pooled corr(thalach, oldpeak)         = {pooled:+.3f}")
print(f"  within no-disease patients          = {within[0]:+.3f}")
print(f"  within disease patients             = {within[1]:+.3f}")
print()
print("why: disease status shifts BOTH variables at once")
print(df.groupby("target")[["thalach", "oldpeak"]].mean().round(2))

**Observe:** a pooled correlation of `−0.348` that is roughly *twice* the within-group
correlations (`−0.190` and `−0.218`), and a group-means table showing disease pushing
`thalach` down (`158.6 → 139.1`) while pushing `oldpeak` up (`0.60 → 1.59`).
**Infer:** most of the pooled association is not a relationship between the two
variables at all — it is disease status moving both in opposite directions, which
manufactures a negative correlation between them even in a world where they are
unrelated within each group. Here it inflates rather than reverses, but the mechanism
is identical to the textbook reversal, and it is the reason "controlling for" a
confounder is standard practice. The failure this prevents downstream is concrete: a
model given `thalach` and `oldpeak` and asked which matters more will produce
coefficients reflecting this shared confounding, not two independent effects — which is
also why Session 9 reads multivariate coefficients as "holding the others fixed" rather
than as standalone effects.

## Step 8 — What correlation cannot tell you

Collecting the failure modes into a checklist, since the coefficient itself carries
none of these warnings.

In [ ]:
checklist = pd.DataFrame([
    ("r near 0",          "no LINEAR relationship",   "a U-shape gives r = 0 too",           "plot it (Step 5)"),
    ("r large",           "strong linear association","could be a few outliers",             "compare Spearman (Step 4)"),
    ("r large",           "the two move together",    "says nothing about which causes which","design, not statistics"),
    ("two inputs r high", "they are redundant",       "pairs miss 3-way redundancy",         "VIF (Step 6)"),
    ("pooled r",          "the overall relationship", "may misstate every subgroup",         "split by confounder (Step 7)"),
    ("r on coded categories", "nothing reliable",     "depends on arbitrary code order",     "chi-square (Session 8)"),
], columns=["when you see", "it means", "but", "check with"])
print(checklist.to_string(index=False))

**Observe:** six rows, each pairing a reading of the coefficient with the specific way
that reading can fail and the diagnostic that catches it.
**Infer:** the third row is the one with the largest real-world cost. Nothing in this
session — or in Sessions 6-8, which only add p-values — can establish that a
relationship is causal; that comes from study design (randomisation, or a longitudinal
design with temporal ordering), and this registry is cross-sectional, so it cannot
support causal claims about any of the correlations found above. That constraint
carries into the deployed system: `oldpeak` correlating with disease means it is useful
for *predicting* who has disease, not that reducing a patient's ST depression would
reduce their risk. A screening tool that gets read as a treatment recommendation is a
failure of exactly this distinction.

## What this session hands to the next one

- **A ranked shortlist of predictors** — `thal`, `ca`, `oldpeak`, `thalach`, `exang`,
  `cp` — which becomes Session 9's candidate input set.
- **A clean multicollinearity verdict** on the continuous inputs (all VIF < 1.4), so
  Session 9's coefficients can be read at face value.
- **Specific hypotheses to test**: is the `thalach` gap between outcome groups real
  (Session 7's t-test)? Is the `cp` association real (Session 8's chi-square)?
- **The correlation-is-not-causation constraint**, which bounds what the finished
  system may claim.

## Try it yourself

1. Recompute Step 2's ranking using Spearman instead of Pearson. Which inputs move
   most, and does Session 3's skewness table predict which ones would?
2. Drop the four |z| > 3 cholesterol patients from Session 2 and re-run Step 5's middle
   panel. How much of `age ~ chol`'s r survives?
3. Add `ca` and `thal` to the VIF table in Step 6. Does treating category codes as
   numbers create redundancy that is not really there?
4. Repeat Step 7 for `age ~ chol`, splitting by `sex`. Does the pooled correlation
   overstate, understate, or fairly represent the within-group ones?